# test — 自动化数据采集（pipeline 测试）

配套 `tool/uload.ipynb` 生成的 `cfg_<kind>_u.._m.._s<seed>.json`：
对每份 cfg，以 workers=m 起 `bin/client` + `bin/server` 灌 cfg，跑 `dur_s` 秒 → 终止 →
把 client 导出的 trace（`temp/tracing.csv`）搬到结果目录。**只搬运原始 trace，不算指标**；
**默认每份测一轮(trials=1)**。

## 结果存放（镜像输入目录）
- 输入目录 = `cfg_dir`（默认 `tool/uload/`，可递归含子目录）；
- 结果根 = `test_dir`（默认 `tool/test/`），镜像 `cfg_dir` 的子目录结构；
- 每份 cfg → 同目录 `trace_<cfg名去cfg_前缀>.csv`；`trials>1` 加 `_r<rep>`。

## temp（client 导出中间目录）
- client 把 trace 写 `./temp/tracing.csv`（宏 `FINS_EXPORT_TRACING_PATH`，相对启动 cwd=仓库根；
  `dag.json` 同目录）；client 启动时会自建该目录；
- notebook 用 `temp_dir` 指向它（相对仓库根或绝对），跑前清理残留、跑后搬走并删原文件。

In [34]:
# ==== 库体：test() —— 跑 cfg → 把 temp/tracing.csv 搬到 out（镜像子目录，与 cfg 同名）====
import os, re, time, signal, subprocess

CFG_RE = re.compile(r"cfg_(?P<kind>\w+)_u(?P<u>\d+)_m(?P<m>\d+)_s(?P<seed>\d+)\.json$")

def repo_root():
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, "bin", "client")) and os.path.isdir(os.path.join(d, "tool")):
            return d
        p = os.path.dirname(d)
        if p == d: return None
        d = p

def find_cfgs(directory):
    out = []
    for rt, _, fns in os.walk(directory):
        for fn in sorted(fns):
            mt = CFG_RE.match(fn)
            if mt:
                out.append((os.path.relpath(os.path.join(rt, fn), directory),
                            int(mt["u"]) / 100.0, int(mt["m"]),
                            mt["kind"], int(mt["seed"])))
    return out

def run_and_move(cfg_path, workers, dur_s, warm_s, port, dest, root, cores=None, temp_dir="temp"):
    """起 client+server 灌 cfg → 跑 dur_s → 终止 → 把 temp 里 client 导出的 trace 搬 dest。
    client 现把 trace 写 ./temp/tracing.csv（宏 FINS_EXPORT_TRACING_PATH，相对启动 cwd=仓库根；
    dag.json 同理写 ./temp/），temp_dir 即该导出目录（仓库根下相对名/绝对路径）。client 自建目录，
    这里仍先建(幂等)并清残留，避免把上一轮的旧 trace 搬给失败轮次。
    cores=None → 裸 client（开发自检）；cores 如 "1-6" → sudo tool/agent.sh <cores> <workers>（需 root/密码）。
    以独立进程组启动，结束发 SIGTERM 整组；独占模式尽力回收 cpuset 分区。"""
    export_dir = temp_dir if os.path.isabs(temp_dir) else os.path.join(root, temp_dir)
    os.makedirs(export_dir, exist_ok=True)
    cands = [os.path.join(export_dir, "tracing.csv"),
             os.path.join(root, "tracing.csv")]         # 旧二进制：导出根目录（兜底）
    for c in cands:
        try: os.remove(c)
        except FileNotFoundError: pass
    if cores:
        cmd = ["sudo", os.path.join(root, "tool", "agent.sh"), str(cores), str(workers)]
    else:
        cmd = [os.path.join(root, "bin", "client"), str(port), os.path.join(root, "lib"),
               str(workers)]   # 插件 .so 现生成在 lib/
    cl = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                          cwd=root, start_new_session=True)
    try:
        time.sleep(warm_s)
        subprocess.run([os.path.join(root, "bin", "server"), cfg_path, str(port)], check=False,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, cwd=root)
        time.sleep(dur_s)
        os.killpg(cl.pid, signal.SIGTERM)          # 整组终止(run.sh/sudo→client)
        try:
            cl.wait(timeout=25)
        except subprocess.TimeoutExpired:
            os.killpg(cl.pid, signal.SIGKILL); cl.wait()
    except Exception:
        try: os.killpg(cl.pid, signal.SIGKILL)
        except Exception: pass
        if cores:
            subprocess.run(["sudo", os.path.join(root, "tool", "agent.sh"), "-r"],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, cwd=root)
        return False
    for _ in range(30):                        # 等 client 落盘再搬（只搬运，不计算指标）
        for src in cands:
            if os.path.exists(src):
                try:
                    os.replace(src, dest)
                    return True
                except (FileNotFoundError, PermissionError):
                    time.sleep(0.1)
        time.sleep(0.1)
    return False

def test(directory, out, temp_dir="temp", dur_s=5.0, warm_s=1.0, trials=1,
         only=None, u=None, m=None, port=18080, cores=None):
    """主入口：对 directory 里每份 cfg 跑一轮(dur_s 可配)，把 temp_dir 里的 tracing.csv 搬到
    out/<镜像子目录>/trace_<cfg名去cfg_前缀>.csv（trials>1 加 _r<rep>）。cores=None 裸 client；
    cores="1-6" 走 sudo tool/run.sh 独占核。返回成功搬了几份。"""
    root = repo_root()
    if root is None: raise RuntimeError("找不到仓库根(含 client 与 tool/)")
    directory = directory if os.path.isabs(directory) else os.path.join(root, directory)
    out = out if os.path.isabs(out) else os.path.join(root, out)
    cfgs = find_cfgs(directory)
    if only: cfgs = [c for c in cfgs if c[3] in only]
    if u is not None: cfgs = [c for c in cfgs if abs(c[1] - u) < 1e-9]
    if m is not None: cfgs = [c for c in cfgs if c[2] == m]
    if not cfgs:
        print(f"[test] {directory} 下没找到 cfg_*.json（先到 tool/uload.ipynb 生成）")
        return 0
    moved = 0
    for rel, uu, mm, kind, seed in cfgs:
        base = os.path.splitext(os.path.basename(rel))[0]
        rdir = os.path.dirname(rel)
        d = out if rdir == "." else os.path.join(out, rdir)
        os.makedirs(d, exist_ok=True)
        stem = base[4:] if base.startswith("cfg_") else base   # 去掉 cfg_ 前缀
        for rep in range(1, trials + 1):
            suffix = "" if trials == 1 else f"_r{rep}"
            dest = os.path.join(d, "trace_" + stem + suffix + ".csv")   # trace_xxx.csv
            ok = run_and_move(os.path.join(directory, rel), mm, dur_s, warm_s, port, dest, root,
                              cores=cores, temp_dir=temp_dir)
            print(f"[test] {kind:9s} u={uu} m={mm} seed={seed} r{rep} -> "
                  + ("moved " + os.path.relpath(dest) if ok else "no_trace"))
            moved += int(ok)
    print(f"[test] 完成: 搬了 {moved}/{len(cfgs) * trials} 份 trace → {out}")
    return moved


### 用法
`test(directory, out='tool/test', temp_dir='temp', dur_s=5.0, warm_s=1.0, trials=1, only, u, m, port, cores=None)`：
- 对每份 cfg：起 client → 预热 `warm_s` → `server` 灌 cfg → 跑 **`dur_s` 秒** → SIGTERM →
  把 client 在 `temp_dir` 导出的 `tracing.csv` 搬到 `out` 镜像子目录为 `trace_<cfg名去cfg_前缀>.csv`；
- **独占核(正式实验)：`cores="1-6"` → 走 `sudo tool/run.sh <cores> <workers>`**（需要 root；jupyter 里要
  passwordless sudo，否则请在终端 `sudo` 跑驱动脚本）；`cores=None` → 裸 `bin/client`，仅开发自检；
- `directory/out/temp_dir` 相对路径按仓库根解析；client 启动 cwd=仓库根 → 导出目录固定 `root/<temp_dir>`。

In [35]:
# ── 参数（可配置）────────────────────────────
cfg_dir   = "tool/uload/"   # 目标 json(cfg)存放目录
test_dir = "tool/test/"          # 结果根(镜像 cfg_dir 子目录结构)
temp_dir = "temp/"              # client 导出 trace 临时目录(旧 tmp/traing.csv)
dur_s     = 5.0                  # ★ 测试时长(秒)：每份 cfg 跑多久
warm_s    = 1                  # 启动 client 后/计时前预热
trials    = 1                    # 每份测几轮(默认 1)

# ── 跑并搬运：起 client/server→跑 dur_s→终止→把 temp/tracing.csv 移到 test 镜像，
#    每个 cfg 生成与它同名的 trace_*.csv（纯搬运，不算指标）────────
test(cfg_dir, test_dir, temp_dir, dur_s=dur_s, warm_s=warm_s, trials=trials)

[test] fork      u=0.2 m=6 seed=10011 r1 -> moved test/trace_fork_u20_m6_s10011.csv
[test] 完成: 搬了 1/1 份 trace → /home/jenny/Documents/GitHub/fins/tool/test/


1